# trainer-subclass-extend — worked example 3: Subclass trainer that adds linear LR warmup via super()._step pattern

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `trainer-subclass-extend`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Learning rate warmup linearly increases the LR from near zero to the target over the first `warmup_steps` gradient updates. This prevents large early updates from destabilizing the model. In the subclass pattern, we override `fit` to call `scheduler.step()` after each batch during warmup, while calling `super()._step()` and `super().validate()` unchanged.

## Worked solution

**Step 1 – Set up a warmup scheduler.** In `__init__`, we call `super().__init__(...)` then build a `LambdaLR` scheduler that returns `min(1.0, step / warmup_steps)` for the first `warmup_steps` steps and 1.0 thereafter.

**Step 2 – Override `fit` minimally.** We need to call `scheduler.step()` after each optimizer step. Instead of duplicating the whole loop, we override `fit` and call `self._step()` (the base's forward+loss method) and `self.validate()` (inherited). The extra line is just `self.scheduler.step()`.

**Step 3 – Track LR per step.** We log `optimizer.param_groups[0]['lr']` to `self.lr_history` after each step so the warmup ramp is visible.

**Step 4 – `_step` and `validate` are inherited.** We don't override them at all. Python's MRO ensures `super()._step(x, y)` reaches the base's forward+loss.

In [ ]:
import torch as t
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

class BaseTrainer3:
    def __init__(self, model, optimizer, train_loader, val_loader, loss_fn):
        self.model = model
        self.optimizer = optimizer
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.loss_fn = loss_fn
        self.step = 0
        self.history = {'train_loss': [], 'val_loss': []}

    def _step(self, x, y):
        return self.loss_fn(self.model(x), y)

    def fit(self, n_epochs):
        for _ in range(n_epochs):
            self.model.train()
            for x, y in self.train_loader:
                loss = self._step(x, y)
                loss.backward()
                self.optimizer.step()
                self.optimizer.zero_grad()
                self.step += 1
                self.history['train_loss'].append(loss.item())
            self.validate()

    def validate(self):
        self.model.eval()
        total, count = 0.0, 0
        with t.inference_mode():
            for x, y in self.val_loader:
                loss = self.loss_fn(self.model(x), y)
                total += loss.item() * x.shape[0]
                count += x.shape[0]
        self.history['val_loss'].append(total / count)


class WarmupTrainer(BaseTrainer3):
    """Extends BaseTrainer3 with linear LR warmup."""
    def __init__(self, model, optimizer, train_loader, val_loader, loss_fn, warmup_steps=4):
        super().__init__(model, optimizer, train_loader, val_loader, loss_fn)
        self.warmup_steps = warmup_steps
        self.scheduler = t.optim.lr_scheduler.LambdaLR(
            self.optimizer,
            lr_lambda=lambda s: min(1.0, (s + 1) / warmup_steps)
        )
        self.lr_history = []

    def fit(self, n_epochs):
        for _ in range(n_epochs):
            self.model.train()
            for x, y in self.train_loader:
                loss = self._step(x, y)  # inherited from base
                loss.backward()
                self.optimizer.step()
                self.scheduler.step()   # warmup extension
                self.optimizer.zero_grad()
                self.step += 1
                self.history['train_loss'].append(loss.item())
                self.lr_history.append(self.optimizer.param_groups[0]['lr'])
            self.validate()  # inherited from base

# Demo
t.manual_seed(6)
X = t.randn(20, 2)
Y = X[:, 0:1]
train_dl = DataLoader(TensorDataset(X[:16], Y[:16]), batch_size=4)
val_dl = DataLoader(TensorDataset(X[16:], Y[16:]), batch_size=4)
model = nn.Linear(2, 1)
opt = t.optim.SGD(model.parameters(), lr=0.1)
trainer = WarmupTrainer(model, opt, train_dl, val_dl, nn.MSELoss(), warmup_steps=4)
trainer.fit(2)
print('lr per step:', [f'{v:.4f}' for v in trainer.lr_history])